<a href="https://colab.research.google.com/github/iZevro/ITCS-3162/blob/Lab-3/lab_03_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — Exercise: Wrangling the Google Play Store Dataset

**ITCS 3162 — Introduction to Data Mining**

**Name:** Khai Robert
**Date:** 5/31/26

This dataset is *messier* than diamonds — it's scraped from the Google Play Store and has the kinds of problems you'll see in the wild: numbers stored as strings, units mixed with values, duplicates, weird placeholder strings, and missing values. That's the point: you'll do real cleaning.

**Source:** Kaggle's "Google Play Store Apps" dataset (mirrored on GitHub for stable URL access).

When you're done, **Restart & Run All**, download as `.ipynb`, and submit via Canvas.


## Setup

Run this cell. It loads the dataset directly from a GitHub URL.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/sumitgirwal/google-play-store-data-analysis/master/googleplaystore.csv"
df = pd.read_csv(URL)
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## Exercise 1 — Initial inspection (10 pts)

In the cell below, print **all three**:
1. The shape of the DataFrame (rows, columns)
2. The output of `df.info()`
3. The output of `df.describe(include="all")`

Then in the markdown cell, answer the questions.


In [2]:
# TODO: print shape, info(), describe(include='all')
print("Shape (rows, cols):", df.shape)
df.info()
df.describe()

Shape (rows, cols): (10841, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


,Rating
count,9367.000000
mean,4.193338
std,0.537431
min,1.000000
25%,4.000000
50%,4.300000
75%,4.500000
max,19.000000


**Answer the following:**

1. How many rows and columns does the dataset have?
- The dataset has 10841 rows and 13 columns.
2. Look at the `Reviews`, `Size`, `Installs`, and `Price` columns. What dtype does pandas assign them? Why is that a problem for a numeric column like `Reviews`?
- The data type is *object*. This data type is not an accurate representation of the recorded information. Instead, they should be *int64* or *float64* where the data can then be better described.
3. Which columns appear to have missing values?
- Columns: #2,#6,#8,#11,#12.



## Exercise 2 — Missing values (10 pts)

In the cell below, compute (a) the count of missing values per column, sorted descending, and (b) the **percentage** of missing values per column rounded to 2 decimal places. Then drop any row that is missing the `Rating` column (since it's the most important missingness to handle) and store the result in `df_r`.


In [3]:
# TODO: counts of missing values per column (sorted descending)
missing_values = df.isna().sum().sort_values(ascending=False)
print("Missing Values:\n", missing_values, "\n")

# TODO: percentage missing per column (rounded to 2 decimal places)
missing_percentage = (df.isna().mean()*100).round(2)
print("Missing Percentage:\n", missing_percentage, "\n")

# TODO: create df_r by dropping rows where Rating is NaN; print its new shape
df_r = df.dropna(subset=["Rating"])
print("New shape (rows, cols):", df_r.shape)

Missing Values:
 Rating            1474
Current Ver          8
Android Ver          3
Content Rating       1
Type                 1
Size                 0
Reviews              0
Category             0
App                  0
Price                0
Installs             0
Last Updated         0
Genres               0
dtype: int64 

Missing Percentage:
 App                0.00
Category           0.00
Rating            13.60
Reviews            0.00
Size               0.00
Installs           0.00
Type               0.01
Price              0.00
Content Rating     0.01
Genres             0.00
Last Updated       0.00
Current Ver        0.07
Android Ver        0.03
dtype: float64 

New shape (rows, cols): (9367, 13)


## Exercise 3 — Duplicates (10 pts)

There are duplicate rows in this dataset (the same app appears multiple times). In the cell below:
1. Print how many fully-duplicate rows exist in `df_r`.
2. How many duplicate values are there in the `App` column specifically (apps appearing under different categories)?
3. Create `df_dedup` by dropping fully-duplicate rows and print its new shape.


In [4]:
# TODO: fully duplicate rows
dups = df.duplicated().sum()
print(f"Duplicate rows: {dups}")

# TODO: duplicate App names (use df_r["App"].duplicated().sum())
app_dups = df_r["App"].duplicated().sum()
print(f"Duplicate app names: {app_dups}")

# TODO: df_dedup = df_r.drop_duplicates(); print shape
df_dedup = df_r.drop_duplicates().copy()
print(f"After dedup: {df_dedup.shape}")

Duplicate rows: 483
Duplicate app names: 1170
After dedup: (8893, 13)


## Exercise 4 — Cleaning the `Installs` column (15 pts)

The `Installs` column looks like `"1,000,000+"` — a string with a `+` and commas. Convert it to an integer column. Steps:

1. Inspect a few unique values: `df_dedup["Installs"].unique()[:10]`
2. Remove `+` and `,` characters with `.str.replace()`
3. Convert to int with `.astype(int)` (or use `pd.to_numeric` with `errors="coerce"` if any values look weird)
4. Assign the cleaned values back to `df_dedup["Installs"]`
5. Verify with `df_dedup["Installs"].dtype` and `.head()`


In [5]:
# TODO: inspect unique values
print("Unique values")
print(df_dedup["Installs"].unique()[:10])

# TODO: clean Installs to integer
df_dedup["Installs"] = df_dedup["Installs"].str.replace("+", "", regex=False)
df_dedup["Installs"] = df_dedup["Installs"].str.replace(",", "", regex=False)
df_dedup["Installs"] = pd.to_numeric(df_dedup["Installs"], errors="coerce")

print("\nstr to num:", df_dedup["Installs"].dtype)
print(df_dedup["Installs"].head())

Unique values
['10,000+' '500,000+' '5,000,000+' '50,000,000+' '100,000+' '50,000+'
 '1,000,000+' '10,000,000+' '5,000+' '100,000,000+']

str to num: float64
0       10000.0
1      500000.0
2     5000000.0
3    50000000.0
4      100000.0
Name: Installs, dtype: float64


## Exercise 5 — Cleaning the `Price` column (15 pts)

`Price` looks like `"0"` for free apps and `"$4.99"` for paid ones. Convert it to a float.

1. Use `.str.replace("$", "", regex=False)` then `.astype(float)`.
2. Watch for any unexpected values (the real dataset has at least one weird row — the column may have a non-numeric placeholder that `pd.to_numeric(..., errors="coerce")` handles gracefully).
3. After cleaning, print how many paid apps there are (where `Price > 0`).


In [6]:
# TODO: clean Price to float
df_dedup["Price"] = df_dedup["Price"].str.replace("$", "", regex=False)
df_dedup["Price"] = pd.to_numeric(df_dedup["Price"], errors="coerce")

# TODO: count paid apps
print(f"Paid apps: {(df_dedup["Price"]>0).sum()}")

Paid apps: 613


## Exercise 6 — Filtering and a derived column (15 pts)

1. Create `top_apps`, containing only apps with `Rating >= 4.5` and `Installs >= 1_000_000`.
2. Print its shape and the top 5 categories of those apps by count.
3. In `df_dedup`, add a new column `Revenue` defined as `Price * Installs` (zero for free apps).
4. Show the top 10 apps by `Revenue`.


In [7]:
# TODO: top_apps and category counts
top_apps = df_dedup[(df_dedup["Rating"] >= 4.5) & (df_dedup["Installs"] >= 1_000_000)]

print("top_apps shape:", top_apps.shape)
print("\nTop 5 categories by count:")
print(top_apps["Category"].value_counts().head(5))

# TODO: Revenue column and top 10
df_dedup["Revenue"] = df_dedup["Price"] * df_dedup["Installs"]

print("\nTop 10 apps by Revenue:")
top_10_revenue = df_dedup.sort_values(by="Revenue", ascending=False)[["App", "Revenue"]].head(10)
print(top_10_revenue)


top_apps shape: (1234, 13)

Top 5 categories by count:
Category
GAME                  272
FAMILY                171
TOOLS                  89
HEALTH_AND_FITNESS     82
PRODUCTIVITY           62
Name: count, dtype: int64

Top 10 apps by Revenue:
                                App     Revenue
4347                      Minecraft  69900000.0
2241                      Minecraft  69900000.0
5351                      I am rich  39999000.0
5356              I Am Rich Premium  19999500.0
4034                  Hitman Sniper   9900000.0
7417  Grand Theft Auto: San Andreas   6990000.0
2883            Facetune - For Free   5990000.0
5578        Sleep as Android Unlock   5990000.0
8804            DraStic DS Emulator   4990000.0
4367       I'm Rich - Trump Edition   4000000.0


## Exercise 7 — Group-by summary (15 pts)

Using `df_dedup`, produce a single summary table with one row per **Category**, containing:
- `n_apps` — number of apps in that category
- `mean_rating` — average rating (rounded to 2 decimals)
- `mean_installs` — average installs (rounded to 0 decimals)
- `total_revenue` — sum of `Revenue`

Sort by `total_revenue` descending. Show only the top 10 rows.


In [8]:
# TODO: groupby summary
summary = df_dedup.groupby("Category").agg(
    n_apps=("App", "count"),
    mean_rating=("Rating", "mean"),
    mean_installs=("Installs", "mean"),
    total_revenue=("Revenue", "sum")
)

summary["mean_rating"] = summary["mean_rating"].round(2)
summary["mean_installs"] = summary["mean_installs"].round(0)

top_10_summary = summary.sort_values(by="total_revenue", ascending=False).head(10)

print(top_10_summary)

                 n_apps  mean_rating  mean_installs  total_revenue
Category                                                          
FAMILY             1718         4.19      5844663.0   1.857743e+08
LIFESTYLE           305         4.10      1753250.0   5.758394e+07
GAME               1074         4.28     29370449.0   4.098684e+07
FINANCE             317         4.13      2430008.0   2.572664e+07
PHOTOGRAPHY         304         4.18     31977773.0   8.941050e+06
MEDICAL             302         4.18       139612.0   8.371355e+06
PERSONALIZATION     310         4.33      6691461.0   7.786310e+06
TOOLS               734         4.05     15600442.0   5.462910e+06
SPORTS              286         4.23      5344516.0   4.706154e+06
PRODUCTIVITY        334         4.20     37314581.0   4.304452e+06


## Exercise 8 — Reflection (10 pts)

In 4–6 sentences, answer all of:

1. Which cleaning step would have been impossible to skip if you wanted to compute average install counts? Why?
- Changing the install count to a number. An object dtype is not appropriate for calculating averages.  
2. What's one piece of information in the raw data that you *could* extract with more work but didn't (e.g., the `Size` column, the `Last Updated` column)? How would you approach it?
- The `size` column could be extracted with more work. I would approach the problem with the same solution used on installs, then adjust the column name so It's understood that the data is measured in mb.
3. Did anything in the data surprise you?
- I was genuinely surprised by how much revenue games are making and how many exist compared to other app categories.


## Submission checklist

- [x] Name and date filled in
- [x] All TODO cells completed and run
- [x] All `YOUR ANSWER` prompts replaced
- [x] **Restart & Run All** completes without errors
- [x] Downloaded as `.ipynb` and uploaded to Canvas
